# 05 · Fleet heat — why the client machine gets hot when the work is elsewhere

The complaint this notebook exists to settle: *the laptop goes angry even when
almost everything runs on the compute host.* That is an attribution question,
and it has exactly two candidate answers — either work is crossing the wire and
landing on the client, or the client is burning CPU on its own account.

**Resource priority on the client host is MEMORY first, then CPU, then space.**
The verdict below is ordered to match, and memory findings are never traded off
against a CPU improvement.

⛔ **`ps %CPU` is a lifetime average** — total CPU over total process age — so a
process that pinned a core an hour ago and has slept since still reads as busy.
Everything here uses the `ytop` probe's sampled `/proc` delta instead, which is
what `htop` actually does.

⛔ **A few boosted cores are enough to reach package temperature.** "14 of 16
cores idle" does **not** show that the heat is not ours; a 2-core load at boost
clocks will sit at the thermal ceiling. Attribute by *share of busy CPU*, never
by count of idle cores.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
import json as _json

HOSTS = H.HOSTS
print("hosts:", HOSTS, " gui host:", H.GUI_HOST)

def probe(host):
    try:
        return _json.loads(H.run_on(host, ["ytop", "--probe"]))
    except (H.ProbeError, _json.JSONDecodeError) as e:
        print(f"  {host}: probe unavailable — {e}")
        return None

probes = {h: probe(h) for h in HOSTS}
probes = {h: p for h, p in probes.items() if p}

In [ ]:
# Temperature is not in the ytop probe, so read hwmon directly. An unreadable
# sensor is a verdict about the sensor: it must not become a comfortable zero.
TEMP_CMD = ("for h in /sys/class/hwmon/hwmon*; do n=$(cat $h/name 2>/dev/null); "
            "for f in $h/temp*_input; do [ -r \"$f\" ] && "
            "echo \"$n $(basename $f) $(cat $f)\"; done; done 2>/dev/null; true")

def temps(host):
    try:
        out = H.run_on(host, ["sh", "-c", TEMP_CMD])
    except H.ProbeError:
        return None
    found = {}
    for line in out.splitlines():
        parts = line.split()
        if len(parts) == 3 and parts[2].lstrip("-").isdigit():
            found[f"{parts[0]}/{parts[1]}"] = int(parts[2]) / 1000.0
        # k10temp / coretemp / acpitz are the package-level candidates
    return found or None

host_temps = {h: temps(h) for h in probes}

def package_temp(found):
    """Hottest package-level sensor, or None. Never defaults to zero."""
    if not found:
        return None
    pkg = [v for k, v in found.items() if k.split("/")[0] in ("k10temp", "coretemp", "acpitz")]
    return max(pkg) if pkg else max(found.values())

for h, found in host_temps.items():
    pt = package_temp(found)
    print(f"{h:10} package {pt if pt is None else f'{pt:.0f} C'}  "
          + ("  ".join(f"{k}={v:.0f}" for k, v in sorted(found.items())) if found else "(no readable sensor)"))

In [ ]:
# FAN AND SOCKET POWER.
#
# ⛔ There is no fan tachometer on the reference client — measured, not assumed:
# the ACPI fan reports 0 forever and both `type=Fan` cooling devices are pinned
# at 1 of 1. Two interfaces that look exactly like fan telemetry and carry none.
# A zero from a stub is reported as ABSENT here, because a column of confident
# zeroes plots beautifully and answers nothing.
#
# Socket power IS live, and sustained socket power plus package temperature is
# what a laptop fan curve actually follows. It is a PROXY and is labelled as one.
POWER_CMD = ("for h in /sys/class/hwmon/hwmon*; do n=$(cat $h/name 2>/dev/null); "
             "[ -r \"$h/power1_average\" ] && echo \"P $n $(cat $h/power1_average)\"; "
             "[ -r \"$h/fan1_input\" ] && echo \"F $n $(cat $h/fan1_input)\"; done 2>/dev/null; true")

def power_and_fan(host):
    try:
        out = H.run_on(host, ["sh", "-c", POWER_CMD])
    except H.ProbeError:
        return None, None
    watts, rpm = None, None
    for line in out.splitlines():
        parts = line.split()
        if len(parts) != 3:
            continue
        kind, _name, raw = parts
        if not raw.lstrip("-").isdigit():
            continue
        if kind == "P":
            watts = int(raw) / 1_000_000.0
        elif kind == "F" and int(raw) > 0:
            rpm = int(raw)          # a stub reads 0; absent beats a fake zero
    return watts, rpm

power = {h: power_and_fan(h) for h in probes}
for h, (w, r) in power.items():
    print(f"{h:10} socket power {'unknown' if w is None else f'{w:5.1f} W'}   "
          f"fan {'no tachometer on this hardware' if r is None else f'{r} rpm'}")

In [ ]:
rows = []
for h, p in probes.items():
    mt, ma = p["mem_total_kb"], p["mem_available_kb"]
    st, sf = p.get("swap_total_kb", 0), p.get("swap_free_kb", 0)
    rows.append({
        "host": h,
        "cores": p["cpu_count"],
        "busy_cores": p["cpu_busy_pct"] / 100.0,
        "busy_pct_of_box": p["cpu_busy_pct"] / (p["cpu_count"] * 100.0) * 100,
        "mem_used_gib": (mt - ma) / 1048576,
        "mem_pct": (mt - ma) / mt * 100,
        "swap_used_gib": (st - sf) / 1048576 if st else 0.0,
        "pkg_temp_c": package_temp(host_temps.get(h)),
        "procs": p["procs_total"],
    })
print(H.table(rows, ["host", "cores", "busy_cores", "busy_pct_of_box", "mem_used_gib",
                     "mem_pct", "swap_used_gib", "pkg_temp_c", "procs"]))

In [ ]:
# ATTRIBUTION — of the CPU that is actually busy, how much is yggterm's own?
OURS = ("yggterm", "WebKitWebProces", "WebKitNetworkPr", "WebKitGPUProces")

attribution = {}
for h, p in probes.items():
    busy = p["cpu_busy_pct"]
    ours = sum(t["cpu_pct"] for t in p["top"] if any(t["comm"].startswith(o) for o in OURS))
    accounted = sum(t["cpu_pct"] for t in p["top"])
    attribution[h] = {"busy_pct": busy, "ours_pct": ours, "accounted_pct": accounted,
                      "our_share": ours / busy if busy else None}
    print(f"\n{h}: {busy:.0f}% busy total, top-N accounts for {accounted:.0f}%")
    print(f"  yggterm + webkit = {ours:.0f}%  =>  {ours/busy*100 if busy else 0:.0f}% of all busy CPU")
    for t in p["top"][:8]:
        mark = "  <== ours" if any(t["comm"].startswith(o) for o in OURS) else ""
        print(f"    {t['cpu_pct']:6.1f}%  {t['rss_kb']/1024:7.0f} MB  {t['comm']:18}{mark}")

In [ ]:
# The tmpfs question. $XDG_RUNTIME_DIR is RAM, so anything growing there is a
# memory leak on the host whose first priority is memory.
RUNTIME_CMD = ('d=${XDG_RUNTIME_DIR:-/run/user/$(id -u)}; '
               'echo "DIR $d"; du -sk "$d" 2>/dev/null | head -1; '
               'du -sk "$d; true"/* 2>/dev/null | sort -n | tail -5')

for h in probes:
    print(f"\n=== {h} ===")
    try:
        print(H.run_on(h, ["sh", "-c", RUNTIME_CMD]).strip())
    except H.ProbeError as e:
        print(f"  unreadable — {e}")

In [ ]:
# The ytrace discovery index specifically: bounded streams, unbounded registry.
REG_CMD = ('r=${XDG_RUNTIME_DIR:-/run/user/$(id -u)}/ytrace/registry.jsonl; '
           '[ -f "$r" ] && echo "$(stat -c%s "$r") $(wc -l < "$r")" || echo "0 0; true"')
registry = {}
for h in probes:
    try:
        size, lines = H.run_on(h, ["sh", "-c", REG_CMD]).split()
        registry[h] = (int(size), int(lines))
        print(f"{h:10} registry.jsonl {int(size)/1048576:8.1f} MiB  {int(lines):>10,} lines  (tmpfs = RAM)")
    except (H.ProbeError, ValueError):
        registry[h] = None
        print(f"{h:10} registry.jsonl unreadable")

# And what it costs to ask it a question.
import time
for h in probes:
    try:
        t0 = time.time(); H.ytrace(h, "registry"); ms = (time.time() - t0) * 1000
        print(f"{h:10} one `ytrace registry` call: {ms:7.0f} ms")
    except H.ProbeError as e:
        print(f"{h:10} registry verb failed: {e}")

In [ ]:
TEMP_WARN_C, TEMP_FAIL_C = 85.0, 92.0
REGISTRY_WARN_MIB, REGISTRY_FAIL_MIB = 32.0, 128.0
MEM_WARN_PCT, MEM_FAIL_PCT = 80.0, 92.0
OUR_SHARE_WARN = 0.50
# THE FAN CONTRACT (docs/idle-cost-model.md §7b): offsite sessions must cost the
# client host essentially nothing. Socket power is the proxy for "is the fan
# about to spin" — there is no tachometer to grade against.
IDLE_WATTS_WARN, IDLE_WATTS_FAIL = 15.0, 25.0

gui = H.GUI_HOST
v = H.Verdict(f"Fleet heat — client host ({gui})")

grow = next((r for r in rows if r["host"] == gui), None)
if grow is None:
    v.note(H.UNKNOWN, "client host probe", f"{gui} did not answer the ytop probe")
else:
    watts, rpm = power.get(gui, (None, None))
    # MEMORY FIRST — the stated priority order on this host.
    reg = registry.get(gui)
    v.check("ytrace registry resident in tmpfs", (reg[0] / 1048576) if reg else None,
            f"<= {REGISTRY_WARN_MIB:.0f} MiB", warn_over=REGISTRY_WARN_MIB, fail_over=REGISTRY_FAIL_MIB,
            detail="append-only discovery index; tmpfs is RAM")
    v.check("memory in use", grow["mem_pct"], f"<= {MEM_WARN_PCT:.0f}%",
            warn_over=MEM_WARN_PCT, fail_over=MEM_FAIL_PCT)
    v.check("swap in use", grow["swap_used_gib"], "<= 1 GiB", warn_over=1.0, fail_over=6.0)
    # THEN CPU.
    share = attribution.get(gui, {}).get("our_share")
    v.check("yggterm's share of busy CPU", share, f"<= {OUR_SHARE_WARN:.0%} of busy CPU",
            warn_over=OUR_SHARE_WARN, fail_over=0.75,
            detail="a high share means the client is burning its own CPU, not receiving work")
    # THEN THE FAN CONTRACT, via its proxy.
    v.check("socket power (fan proxy)", watts, f"<= {IDLE_WATTS_WARN:.0f} W with offsite work only",
            warn_over=IDLE_WATTS_WARN, fail_over=IDLE_WATTS_FAIL,
            detail="offsite sessions must cost this host ~nothing; onsite processes are "
                   "the user's own and out of scope")
    v.check("package temperature", grow["pkg_temp_c"], f"<= {TEMP_WARN_C:.0f} C",
            warn_over=TEMP_WARN_C, fail_over=TEMP_FAIL_C)
    if rpm is None:
        v.note(H.UNKNOWN, "fan speed",
               "no tachometer on this hardware — the ACPI fan and both `type=Fan` cooling "
               "devices are stubs. Graded on the socket-power proxy instead; see "
               "docs/idle-cost-model.md §7b.")

    if share is not None:
        v.note(H.WARN if share > OUR_SHARE_WARN else H.PASS, "heat attribution",
               (f"{share:.0%} of busy CPU is the GUI and its webview — the client is generating "
                "its own heat; look at the render pipeline (notebook 02), not the wire")
               if share > OUR_SHARE_WARN else
               (f"only {share:.0%} of busy CPU is ours — the heat is coming from somewhere else "
                "on this machine and yggterm is not the mechanism"))
v.show()